# 06 Differentiable features in PyTorch

`afx_audio.torch.functional` implements the STFT, mel spectrogram, MFCC, chroma, GFCC and the spectral descriptors in torch with the same numbers as the NumPy extractors. Inputs are `(time,)` or `(batch, time)` tensors on any device, and gradients flow back to the waveform. `AFXFeatureModule` wraps them as an `nn.Module`.

Needs `pip install "afx-audio[torch]"`; the first cell stops with an install hint without it.

In [ ]:
import importlib.util

if importlib.util.find_spec('torch') is None:
    raise ImportError('This notebook needs PyTorch: pip install "afx-audio[torch]"')

import numpy as np
import matplotlib.pyplot as plt
import torch

import afx_audio as afx
from afx_audio.extractors import cepstral_features
from afx_audio.torch import AFXFeatureModule, supported_features
from afx_audio.torch import functional as F_afx
import demo_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
print(f'torch {torch.__version__} on {device}')
print('torch path:', supported_features())

## Data

One-second clips: four classes from the configured dataset, or the synthetic classes. `load_audio(..., duration=1.0)` reads only the first second; shorter files are zero-padded so clips stack into a batch.

In [ ]:
def one_second(path):
    x, _ = afx.load_audio(path, sr=SR, duration=1.0)
    return np.pad(x, (0, SR - x.size))


labelled = demo_data.labelled_files(n_classes=4, n_per_class=16)
if labelled:
    clips = [one_second(p) for p in labelled[0]]
    labels = np.array(labelled[1])
else:
    clips, labels = demo_data.synthetic_dataset(n_per_class=16, seconds=1.0, sr=SR)
classes = sorted(set(labels.tolist()))
batch = torch.as_tensor(np.stack(clips), dtype=torch.float32, device=device)
print('batch:', tuple(batch.shape), 'classes:', classes)

## Parity with the NumPy path

The same MFCCs from the NumPy extractor and the torch function, in float64.

In [ ]:
signal = clips[0]
mfcc_np = cepstral_features.extract_mfcc(signal.astype(np.float64), SR, n_mfcc=13,
                                         frame_size=1024, hop_length=256)['mfcc']
mfcc_pt = F_afx.mfcc(torch.as_tensor(signal, dtype=torch.float64, device=device), SR, n_mfcc=13,
                     frame_size=1024, hop_length=256).cpu().numpy()
print('shapes', mfcc_np.shape, mfcc_pt.shape, '| max abs difference', np.abs(mfcc_np - mfcc_pt).max())

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, m, title in zip(ax, [mfcc_np, mfcc_pt, np.abs(mfcc_np - mfcc_pt)], ['NumPy', 'torch', '|difference|']):
    im = a.imshow(m, aspect='auto', origin='lower', cmap='magma')
    a.set_title(title)
    a.set_xlabel('frame')
    fig.colorbar(im, ax=a)
ax[0].set_ylabel('MFCC')
fig.tight_layout()

## Batches

Every transform broadcasts over the leading batch axis; this is where a GPU pays off.

In [ ]:
mel = F_afx.melspectrogram(batch, SR, n_mels=64, frame_size=1024, hop_length=256)
centroid = F_afx.spectral_centroid(batch, SR, frame_size=1024, hop_length=256)
print('mel spectrogram', tuple(mel.shape), '(batch, n_mels, frames)')
print('spectral centroid', tuple(centroid.shape), '(batch, frames)')

fig, ax = plt.subplots(1, len(classes), figsize=(3.2 * len(classes), 2.8))
for a, label in zip(ax, classes):
    i = int(np.flatnonzero(labels == label)[0])
    a.imshow(10 * np.log10(mel[i].cpu().numpy() + 1e-10), aspect='auto', origin='lower', cmap='viridis')
    a.set_title(f'{label} (log-mel)')
fig.tight_layout()

## Gradients reach the waveform

Starting from low-level noise, gradient descent on the waveform until its MFCCs match a target clip's.

In [ ]:
target = batch[0]
target_mfcc = F_afx.mfcc(target, SR, n_mfcc=13, frame_size=1024, hop_length=256).detach()

torch.manual_seed(0)
x = (0.01 * torch.randn_like(target)).requires_grad_(True)
opt = torch.optim.Adam([x], lr=1e-2)
losses = []
for step in range(60):
    opt.zero_grad()
    loss = torch.nn.functional.mse_loss(F_afx.mfcc(x, SR, n_mfcc=13, frame_size=1024, hop_length=256), target_mfcc)
    loss.backward()
    opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(losses)
ax.set_xlabel('step')
ax.set_ylabel('MFCC MSE')
ax.set_title(f'{losses[0]:.1f} -> {losses[-1]:.3f}')
fig.tight_layout()

`zero_crossing_rate` and `spectral_rolloff` count or select, so their gradient with respect to the waveform is zero; the other transforms propagate useful gradients.

## `AFXFeatureModule`

Takes a features block like `extract_all_features`. Windows, filterbanks and DCT matrices are buffers, so `.to(device)` moves them. With `output='vector'` it returns `(batch, n_features)` and fits in `nn.Sequential`.

In [ ]:
frame = {'frame_size': 1024, 'hop_length': 256}
frontend = AFXFeatureModule(
    features={'mfcc': {**frame, 'n_mfcc': 13}, 'spectral_centroid': frame, 'spectral_flatness': frame},
    sample_rate=SR, aggregation='mean', output='vector',
).to(device)
vectors = frontend(batch)
print('vectors:', tuple(vectors.shape))
print(frontend)

## A classifier trained through the front end

Half the clips of each class for training, half for testing. `BatchNorm1d` puts the MFCCs (tens to hundreds) and the centroid (kHz) on one scale.

In [ ]:
torch.manual_seed(0)
y = torch.as_tensor([classes.index(c) for c in labels], device=device)
rng = np.random.default_rng(0)
train = np.zeros(len(labels), dtype=bool)
for c in classes:
    idx = np.flatnonzero(labels == c)
    train[rng.permutation(idx)[: idx.size // 2]] = True
train = torch.as_tensor(train, device=device)

model = torch.nn.Sequential(
    frontend,
    torch.nn.BatchNorm1d(vectors.shape[1]),
    torch.nn.Linear(vectors.shape[1], len(classes)),
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for epoch in range(100):
    model.train()
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(batch[train]), y[train])
    loss.backward()
    opt.step()

model.eval()
with torch.no_grad():
    pred = model(batch).argmax(1)
print(f'final training loss {loss.item():.3f}')
print(f'train accuracy {(pred[train] == y[train]).float().mean():.2f}, '
      f'test accuracy {(pred[~train] == y[~train]).float().mean():.2f} (chance {1 / len(classes):.2f})')

PyTorch stays optional: `import afx_audio` works without it, and features not on the torch path remain available through the NumPy API.